# RDMA Fundamentals

Establish a safe, read-only baseline for RDMA device and utility discovery.

## Objectives

Identify available RDMA tools and frame an experiment around devices, ports, and data movement.

## Background

RDMA enables direct data movement with low CPU involvement, but measurements depend on device, link, memory registration, queue, and topology details.

## Prediction

The DGX Spark should expose one or more Mellanox/NVIDIA RDMA-capable devices through the Linux RDMA subsystem.

Before running the experiment, I predict:

1. At least one of `ibv_devices`, `ibstat`, and `rdma` will be installed.
2. `/sys/class/infiniband` will contain one or more RDMA devices associated with the high-speed ConnectX interfaces.
3. Each physical port will report a link layer, administrative state, physical state, and nominal link rate.
4. The RDMA devices will map to Linux network interfaces, but an active physical link does not necessarily imply that an IP address or usable RDMA route is configured.
5. The reported link layer may be Ethernet rather than native InfiniBand. In that case, later RDMA experiments would use RoCE and would depend on Ethernet addressing and configuration.

These are architectural expectations, not measured properties. The experiment below records the actual local configuration.

## Environment

In [2]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [3]:
from common.rdma import detect_rdma_utilities, run_rdma_utility


rdma_utilities = detect_rdma_utilities()

for utility in rdma_utilities:
    status = "available" if utility.available else "missing"
    print(f"{utility.name:<12} {status:<9} {utility.path or '-'}")

ibv_devices  available /usr/bin/ibv_devices
ibstat       missing   -
rdma         available /usr/bin/rdma


In [4]:
def print_command_result(result) -> None:
    command_text = " ".join(result.command)

    print(f"$ {command_text}")
    print(f"return code: {result.returncode}")

    if result.timed_out:
        print("status: timed out")
    elif result.executable_missing:
        print("status: executable missing")
    elif result.error is not None:
        print(f"status: {result.error}")
    else:
        print(f"status: {'succeeded' if result.succeeded else 'failed'}")

    if result.stdout.strip():
        print("\nstdout:")
        print(result.stdout.rstrip())

    if result.stderr.strip():
        print("\nstderr:")
        print(result.stderr.rstrip())

    print()


available_utility_names = {
    utility.name for utility in rdma_utilities if utility.available
}

rdma_command_results = []

if "ibv_devices" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibv_devices"))

if "ibstat" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibstat"))

if "rdma" in available_utility_names:
    rdma_command_results.extend(
        [
            run_rdma_utility("rdma", "dev", "show"),
            run_rdma_utility("rdma", "link", "show"),
        ]
    )

if not rdma_command_results:
    print("No supported RDMA utilities were available.")
else:
    for result in rdma_command_results:
        print_command_result(result)

$ ibv_devices
return code: 0
status: succeeded

stdout:
    device          	   node GUID
    ------          	----------------
    rocep1s0f0      	4cbb470300830241
    rocep1s0f1      	4cbb470300830242
    roceP2p1s0f0    	4cbb470300830245
    roceP2p1s0f1    	4cbb470300830246

$ rdma dev show
return code: 0
status: succeeded

stdout:
0: rocep1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0241 sys_image_guid 4cbb:4703:0083:0241 
1: rocep1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0242 sys_image_guid 4cbb:4703:0083:0241 
2: roceP2p1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0245 sys_image_guid 4cbb:4703:0083:0241 
3: roceP2p1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0246 sys_image_guid 4cbb:4703:0083:0241

$ rdma link show
return code: 0
status: succeeded

stdout:
link rocep1s0f0/1 state DOWN physical_state DISABLED netdev enp1s0f0np0 
link rocep1s0f1/1 state ACTIVE physical_state LINK_UP netdev enp1s0f1np1 
link roceP2p1s0f0/1

### Linux sysfs inventory

Command-line utilities present a user-space view of the RDMA subsystem. Linux also exposes device, port, and network-interface relationships through `/sys/class/infiniband`.

The following cell reads that hierarchy directly so that the inventory remains useful even if some optional RDMA utilities are absent.

In [ ]:
import pandas as pd


def read_text(path: Path) -> str | None:
    try:
        return path.read_text().strip()
    except (FileNotFoundError, PermissionError, OSError):
        return None


infiniband_root = Path("/sys/class/infiniband")
rdma_device_rows = []
rdma_port_rows = []

if not infiniband_root.is_dir():
    print(f"{infiniband_root} does not exist.")
else:
    rdma_devices = sorted(path for path in infiniband_root.iterdir() if path.is_dir())

    if not rdma_devices:
        print(f"No RDMA devices were found under {infiniband_root}.")

    for rdma_device in rdma_devices:
        device_path = rdma_device / "device"
        network_path = device_path / "net"

        network_interfaces = (
            sorted(path.name for path in network_path.iterdir())
            if network_path.is_dir()
            else []
        )

        pci_device = None
        try:
            pci_device = device_path.resolve().name
        except OSError:
            pass

        driver = None
        driver_path = device_path / "driver"
        try:
            if driver_path.exists():
                driver = driver_path.resolve().name
        except OSError:
            pass

        rdma_device_rows.append(
            {
                "rdma_device": rdma_device.name,
                "node_type": read_text(rdma_device / "node_type"),
                "firmware_version": read_text(rdma_device / "fw_ver"),
                "node_guid": read_text(rdma_device / "node_guid"),
                "sys_image_guid": read_text(rdma_device / "sys_image_guid"),
                "pci_device": pci_device,
                "driver": driver,
                "network_interfaces": ", ".join(network_interfaces) or None,
            }
        )

        ports_path = rdma_device / "ports"
        if not ports_path.is_dir():
            continue

        for port_path in sorted(
            ports_path.iterdir(),
            key=lambda path: int(path.name) if path.name.isdigit() else path.name,
        ):
            if not port_path.is_dir():
                continue

            rdma_port_rows.append(
                {
                    "rdma_device": rdma_device.name,
                    "port": port_path.name,
                    "state": read_text(port_path / "state"),
                    "physical_state": read_text(port_path / "phys_state"),
                    "link_layer": read_text(port_path / "link_layer"),
                    "rate": read_text(port_path / "rate"),
                    "lid": read_text(port_path / "lid"),
                    "lid_mask_count": read_text(port_path / "lid_mask_count"),
                    "sm_lid": read_text(port_path / "sm_lid"),
                }
            )

rdma_devices_df = pd.DataFrame(rdma_device_rows)
rdma_ports_df = pd.DataFrame(rdma_port_rows)

print("RDMA devices")
display(rdma_devices_df)

print("\nRDMA ports")
display(rdma_ports_df)

RDMA devices


,rdma_device,node_type,firmware_version,node_guid,sys_image_guid,pci_device,driver,network_interfaces
0,roceP2p1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0245,4cbb:4703:0083:0241,0002:01:00.0,mlx5_core,enP2p1s0f0np0
1,roceP2p1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0246,4cbb:4703:0083:0241,0002:01:00.1,mlx5_core,enP2p1s0f1np1
2,rocep1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0241,4cbb:4703:0083:0241,0000:01:00.0,mlx5_core,enp1s0f0np0
3,rocep1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0242,4cbb:4703:0083:0241,0000:01:00.1,mlx5_core,enp1s0f1np1



RDMA ports


,rdma_device,port,state,physical_state,link_layer,rate,lid,lid_mask_count,sm_lid
0,roceP2p1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
1,roceP2p1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0
2,rocep1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
3,rocep1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0


### Associated network interfaces

RDMA device discovery alone does not show whether the corresponding Linux network interfaces are administratively enabled, physically connected, or assigned addresses.

The next cell reads the ordinary Linux interface state and uses `ip` only for read-only address and route reporting.

In [6]:
from common.utils import run_command


rdma_network_interfaces = sorted(
    {
        interface
        for row in rdma_device_rows
        for interface in (row["network_interfaces"] or "").split(", ")
        if interface
    }
)

network_interface_rows = []

for interface in rdma_network_interfaces:
    interface_path = Path("/sys/class/net") / interface

    network_interface_rows.append(
        {
            "interface": interface,
            "operstate": read_text(interface_path / "operstate"),
            "carrier": read_text(interface_path / "carrier"),
            "mtu": read_text(interface_path / "mtu"),
            "speed_mbps": read_text(interface_path / "speed"),
            "duplex": read_text(interface_path / "duplex"),
            "address": read_text(interface_path / "address"),
        }
    )

network_interfaces_df = pd.DataFrame(network_interface_rows)

print("RDMA-associated network interfaces")
display(network_interfaces_df)

if not rdma_network_interfaces:
    print("No Linux network interfaces were mapped to the RDMA devices.")
else:
    for interface in rdma_network_interfaces:
        print_command_result(
            run_command(("ip", "-details", "address", "show", "dev", interface))
        )

    print_command_result(run_command(("ip", "route", "show")))

RDMA-associated network interfaces


,interface,operstate,carrier,mtu,speed_mbps,duplex,address
0,enP2p1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:45
1,enP2p1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:46
2,enp1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:41
3,enp1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:42


$ ip -details address show dev enP2p1s0f0np0
return code: 0
status: succeeded

stdout:
5: enP2p1s0f0np0: <NO-CARRIER,BROADCAST,MULTICAST,UP> mtu 1500 qdisc mq state DOWN group default qlen 1000
    link/ether 4c:bb:47:83:02:45 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p0 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.0

$ ip -details address show dev enP2p1s0f1np1
return code: 0
status: succeeded

stdout:
6: enP2p1s0f1np1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc mq state UP group default qlen 1000
    link/ether 4c:bb:47:83:02:46 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p1 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.1 
    

## Observations

Run the cells above on one DGX Spark and preserve their outputs before drawing conclusions.

The initial interpretation should answer:

1. Which RDMA utilities are installed?
2. Which RDMA devices are visible?
3. Which kernel driver and PCI device back each RDMA device?
4. Which Linux network interfaces map to those devices?
5. What link layer does each RDMA port report?
6. Which ports are active, and what nominal rate do they report?
7. Are the mapped network interfaces up and carrying an IP address?
8. Does the routing table contain routes through those interfaces?

No bandwidth, latency, CPU-overhead, or GPU-direct conclusions can be made from this inventory alone.

## Explanation

TODO: Relate the observed device and link information to the RDMA data path.

## Connection to LLMs

RDMA supports low-latency transfer of tensors and collective traffic in distributed inference and training.

## Further Exploration

TODO: Define a two-node bandwidth and latency experiment with explicit safety and cleanup steps.